# 🚀 Taller Práctico: El Arte de Explorar Grafos

**Materia:** Estructuras de Datos Dinámicas  
**Monitoría:** Samuel Romaña Acevedo  

---

## 🧠 Un mensaje antes de empezar...
Hoy en día, cualquier Inteligencia Artificial puede generar un algoritmo de Dijkstra o un BFS en 2 segundos. Si copian el enunciado y lo pegan en ChatGPT, tendrán el código listo para entregar. **Pero el parcial no funciona así, y la vida profesional menos.**

Los grafos no se tratan de memorizar ciclos `while` o diccionarios; se trata de **aprender a modelar la realidad**. Saber mapear un problema real (ciudades, virus, redes sociales) en Nodos y Aristas es un superpoder. Cada vez que simulan un algoritmo en papel y entienden por qué la cola (Queue) procesa por niveles y la pila (Stack) profundiza, su cerebro expande su capacidad de abstracción. Hoy vamos a entrenar el modelo mental, no el teclado. ¡Dibujen en papel antes de codificar!

## 📋 Reglas de Juego para hoy:
1. **Cero librerías mágicas.** Prohibido usar `networkx` o similares. Usaremos únicamente listas, diccionarios, `deque` (para colas) y `heapq` (para colas de prioridad).
2. **Respetar la clase base.** No pueden modificar la estructura interna de `GraphL` ni `GraphWL`. Trabajen con lo que estas clases exponen (su `adj_list`).
3. **Control estricto de estado.** Todo nodo visitado debe ser registrado. Un ciclo infinito por un grafo con ciclos es un error imperdonable hoy.

### **Ejecutar esta celda antes de empezar a codificar**

In [1]:
"""Implementaciones base de grafos.

Incluye las cuatro combinaciones de representación / peso:
- GraphL:  lista de adyacencia, sin peso
- GraphM:  matriz de adyacencia, sin peso
- GraphWL: lista de adyacencia, con peso
- GraphWM: matriz de adyacencia, con peso

Todas soportan grafos dirigidos y no dirigidos mediante el parámetro
`directed` de `add_edge` (True por defecto).

Uso:
    from base import GraphL, GraphM, GraphWL, GraphWM
"""
from typing import Any, Dict, List, Tuple


class GraphL:
    """Grafo (sin peso) representado con lista de adyacencia."""

    def __init__(self) -> None:
        self.adj_list: Dict[Any, List[Any]] = {}
        self.size: int = 0

    def add_vertex(self, value: Any) -> None:
        if value in self.adj_list:
            return
        self.adj_list[value] = []
        self.size += 1

    def add_edge(self, vertex_1: Any, vertex_2: Any, directed: bool = True) -> None:
        if vertex_1 not in self.adj_list:
            self.add_vertex(vertex_1)
        if vertex_2 not in self.adj_list:
            self.add_vertex(vertex_2)

        if vertex_2 not in self.adj_list[vertex_1]:
            self.adj_list[vertex_1].append(vertex_2)
        if not directed and vertex_1 not in self.adj_list[vertex_2]:
            self.adj_list[vertex_2].append(vertex_1)

    def __repr__(self) -> str:
        return "\n".join(f"{v}: {ns}" for v, ns in self.adj_list.items())


class GraphM:
    """Grafo (sin peso) representado con matriz de adyacencia."""

    def __init__(self) -> None:
        self.adj_matrix: List[List[int]] = []
        self.nodes: List[Any] = []
        self.size: int = 0

    def add_vertex(self, value: Any) -> None:
        if value in self.nodes:
            return
        self.nodes.append(value)
        for row in self.adj_matrix:
            row.append(0)
        self.adj_matrix.append([0] * len(self.nodes))
        self.size += 1

    def add_edge(self, vertex_1: Any, vertex_2: Any, directed: bool = True) -> None:
        if vertex_1 not in self.nodes:
            self.add_vertex(vertex_1)
        if vertex_2 not in self.nodes:
            self.add_vertex(vertex_2)

        pos_v1 = self.nodes.index(vertex_1)
        pos_v2 = self.nodes.index(vertex_2)

        self.adj_matrix[pos_v1][pos_v2] = 1
        if not directed:
            self.adj_matrix[pos_v2][pos_v1] = 1

    def __repr__(self) -> str:
        header = "   " + " ".join(str(n) for n in self.nodes)
        rows = [
            str(self.nodes[i]) + "  " + " ".join(map(str, row))
            for i, row in enumerate(self.adj_matrix)
        ]
        return "\n".join([header, *rows])


class GraphWL:
    """Grafo ponderado representado con lista de adyacencia."""

    def __init__(self) -> None:
        self.adj_list: Dict[Any, List[Tuple[Any, float]]] = {}
        self.size: int = 0

    def add_vertex(self, value: Any) -> None:
        if value in self.adj_list:
            return
        self.adj_list[value] = []
        self.size += 1

    def add_edge(self, vertex_1: Any, vertex_2: Any, weight: float, directed: bool = True) -> None:
        if vertex_1 not in self.adj_list:
            self.add_vertex(vertex_1)
        if vertex_2 not in self.adj_list:
            self.add_vertex(vertex_2)

        self.adj_list[vertex_1].append((vertex_2, weight))
        if not directed:
            self.adj_list[vertex_2].append((vertex_1, weight))

    def __repr__(self) -> str:
        return "\n".join(f"{v}: {ns}" for v, ns in self.adj_list.items())


class GraphWM:
    """Grafo ponderado representado con matriz de adyacencia."""

    def __init__(self, non_representative_value: float = 0) -> None:
        self.adj_matrix: List[List[float]] = []
        self.nodes: List[Any] = []
        self.size: int = 0
        self.non_representative_value = non_representative_value

    def add_vertex(self, value: Any) -> None:
        if value in self.nodes:
            return
        self.nodes.append(value)
        for row in self.adj_matrix:
            row.append(self.non_representative_value)
        self.adj_matrix.append([self.non_representative_value] * len(self.nodes))
        self.size += 1

    def add_edge(self, vertex_1: Any, vertex_2: Any, weight: float, directed: bool = True) -> None:
        if vertex_1 not in self.nodes:
            self.add_vertex(vertex_1)
        if vertex_2 not in self.nodes:
            self.add_vertex(vertex_2)

        pos_v1 = self.nodes.index(vertex_1)
        pos_v2 = self.nodes.index(vertex_2)

        self.adj_matrix[pos_v1][pos_v2] = weight
        if not directed:
            self.adj_matrix[pos_v2][pos_v1] = weight

    def __repr__(self) -> str:
        header = "   " + " ".join(str(n) for n in self.nodes)
        rows = [
            str(self.nodes[i]) + "  " + " ".join(map(str, row))
            for i, row in enumerate(self.adj_matrix)
        ]
        return "\n".join([header, *rows])


## 🦠 Ejercicio 1: Propagación de un Virus Informático (BFS)

Una red de computadoras ha sufrido una brecha de seguridad. El virus se propaga de una computadora infectada a todas sus conexiones directas en **exactamente 1 minuto**. El grafo de la red se representa mediante `GraphL` (lista de adyacencia sin peso).

Queremos saber el alcance del daño en un momento específico. 

⚠️ **Restricciones:** 
* No puedes usar recursividad. Debes controlar explícitamente el tiempo (o nivel) utilizando una Cola (`deque`).
* Un nodo no puede infectarse dos veces.

---
### 🧪 Ejemplo de prueba:
📥 **Entrada:** `inicio = 'A', T = 2`
(Asumiendo un grafo donde A conecta con B y C; y B conecta con D y E).

✅ **Salida esperada:** `['D', 'E']`
*(En T=0 está 'A'. En T=1 se infectan 'B' y 'C'. En T=2 el virus pasa de 'B' a 'D' y 'E').*

In [ ]:
from collections import deque

def computadoras_infectadas(grafo_l: GraphL, inicio: str, tiempo_t: int) -> list:
    if inicio not in grafo_l:
        return []

    pass

# 🧪 PRUEBA
g = GraphL()
for u, v in [('A','B'), ('A','C'), ('B','D'), ('B','E'), ('C','F')]:
    g.add_edge(u, v, directed=False)

print("Infectados en T=1:", computadoras_infectadas(g, 'A', 1)) # Esperado: ['B', 'C']
print("Infectados en T=2:", computadoras_infectadas(g, 'A', 2)) # Esperado: ['D', 'E', 'F']

## 🕵️ Ejercicio 2: Comunidades Aisladas (DFS)

Estás analizando las interacciones de un foro clandestino, representadas en un grafo no dirigido con la clase `GraphL`. Quieres descubrir cuántos "grupos aislados" de personas existen (usuarios que interactúan entre ellos, pero están totalmente desconectados del resto del foro). Además, necesitas saber el tamaño del grupo más grande.

⚠️ **Restricciones:** 
* Debes garantizar que tu algoritmo pase por **todos** los nodos del grafo, no solo por los que están conectados al nodo 0.
* Usa recursividad (DFS) para profundizar en las conexiones de cada nueva comunidad descubierta.

---
### 🧪 Ejemplo de prueba:
📥 **Entrada:** Grafo con aristas `(1-2), (2-3)` y `(4-5)`. (El nodo 6 no tiene conexiones).

✅ **Salida esperada:** `(3, 3)`
*(Hay 3 comunidades: [1,2,3], [4,5] y [6]. La más grande tiene tamaño 3).*

In [ ]:
def analizar_comunidades(grafo_l: GraphL) -> tuple:
    # Necesitas un set() de visitados, un contador de comunidades y una variable para el tamaño máximo.
    pass

# 🧪 PRUEBA
g2 = GraphL()
for u, v in [(1,2), (2,3), (4,5)]:
    g2.add_edge(u, v, directed=False)
g2.add_vertex(6) # Nodo completamente aislado

print("(Num Comunidades, Tamaño Máx):", analizar_comunidades(g2)) 
# Esperado: (3, 3)

## 🚒 Ejercicio 3: Ruta de Evacuación Segura (Dijkstra)

Durante una emergencia, el equipo de rescate debe trazar la ruta desde la `base` hasta la `zona cero`. El mapa está modelado en `GraphWL` (lista con peso), donde el peso de cada calle representa su nivel de peligro (escombros, fuego, etc.). 

Debes devolver el peligro total acumulado y **la ruta exacta** de nodos a transitar.

⚠️ **Restricciones:** 
* Estrictamente obligatorio usar el algoritmo de Dijkstra con `heapq`.
* Si no hay ruta posible, debes retornar `(-1, [])`.

---
### 🧪 Ejemplo de prueba:
📥 **Entrada:** `base = 'A'`, `destino = 'C'`. 
Rutas: `A -> C (Peligro: 10)`. `A -> B (Peligro: 2)`, `B -> C (Peligro: 3)`.

✅ **Salida esperada:** `(5, ['A', 'B', 'C'])`
*(Aunque A->C es directo, pasar por B acumula menos peligro).*

In [ ]:
import heapq

def ruta_de_rescate(grafo_wl: GraphWL, inicio: str, destino: str) -> tuple:
    pass

# 🧪 PRUEBA
g3 = GraphWL()
g3.add_edge('A', 'C', 10, directed=True)
g3.add_edge('A', 'B', 2, directed=True)
g3.add_edge('B', 'C', 3, directed=True)

print("Ruta óptima:", ruta_de_rescate(g3, 'A', 'C'))
# Esperado: (5, ['A', 'B', 'C'])